# AIRA Model Fine-Tuning — QLoRA SFT on Google Colab
### Autonomous Infrastructure Resilience Architecture

This notebook implements the **Phase 4** SFT (Supervised Fine-Tuning) pipeline for **AIRA**. 
It uses **Unsloth** for 2x faster, memory-efficient QLoRA training on a free-tier Google Colab T4 GPU to fine-tune `gemma-2-2b-it` (or `gemma-2-9b-it`) on our self-generated adversarial trajectory dataset (`sft_dataset.jsonl`).

At the end of training, it exports the weights as a **4-bit GGUF model** for direct integration with **Ollama** in Phase 5.

### 1. Install Unsloth and Dependencies

In [ ]:
%%capture
# Install Unsloth and standard PyTorch dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets pydantic structlog

### 2. Load Model and Tokenizer (4-bit Quantization)

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Supports long Trivy cluster scans and multi-round battle context
dtype = None # Auto-detect Float16 or Bfloat16 based on hardware
load_in_4bit = True # 4-bit quantization reduces VRAM footprint to fit easily on T4 GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-2-2b-it", # Dev baseline model (swap to gemma-2-9b-it for USENIX paper results)
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 3. Configure LoRA Adapters (Rank 64 / Alpha 128)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # LoRA Rank (as specified in PDR3)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128, # LoRA Alpha ratio
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Bypasses PyTorch checkpoint overhead, saving 30% VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 4. Load & Format Trajectory Dataset (`sft_dataset.jsonl`)

In [ ]:
# Upload your 'sft_dataset.jsonl' to the Colab files sidebar before running this cell
from datasets import load_dataset

dataset = load_dataset("json", data_files="sft_dataset.jsonl", split="train")

# Format prompts into Gemma standard chat instruction template
def format_prompts(examples):
    systems = examples["system"]
    users = examples["user"]
    assistants = examples["assistant"]
    texts = []
    for system, user, assistant in zip(systems, users, assistants):
        # Construct target instruct format
        text = f"<bos><start_of_turn>system\n{system}<end_of_turn>\n<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n{assistant}<end_of_turn><eos>"
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(format_prompts, batched = True)

### 5. Training Configuration (TRL SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set to True for significantly faster packing on short context turns
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100, # Adjust depending on desired loss stability
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

### 6. Execute Fine-Tuning

In [ ]:
trainer_stats = trainer.train()

### 7. Export weights to 4-bit GGUF (for local Ollama deployment)

In [ ]:
# Export the fine-tuned model as 4-bit GGUF
# This automatically saves the file to 'gemma-2b-aira-unsloth.Q4_K_M.gguf' in your Colab workspace
model.save_pretrained_merged("gemma-2b-aira-unsloth", tokenizer, save_method = "gguf")

# You can now download this GGUF file and load it on your local Ollama server!